# RSO Reproduction — 100 aa Unconditional Generation

**Goal:** A single 100-aa end-to-end reproduction of the RSO computational workflow (Frank et al. 2024 *Science*) for a single 100-amino-acid monomer: hallucinate a backbone via Relaxed Sequence Optimization (RSO), design 8 sequences with ProteinMPNN (soluble weights, T=0.1, remove Cys), validate each with AlphaFold2 single-sequence model_4_ptm (3 recycles), and compute RMSD / TM-score / mean pLDDT vs. the designed backbone.

**Estimated cost on Colab A100:** ~5–15 min per candidate × 8 ≈ 1–2 h total, dominated by AF2 validation.

**IMPORTANT:** Attach an NVIDIA GPU runtime (A100 40 GB or L4 24 GB preferred; T4 16 GB works for 100 AA). If no GPU is detected, this notebook will STOP before running any compute.

**Reproducibility notes:** fixed seed 42, ColabDesign pinned commit, all provenance written to per-run metadata.json. See docs/provenance.md.


In [ ]:
# --- setup: check GPU + install deps (run once per Colab session) ---
import os, shutil, subprocess, sys, json, time, pathlib, datetime
from pathlib import Path

GPU_AVAILABLE = False
try:
    import subprocess as sp
    r = sp.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
               capture_output=True, text=True, timeout=20)
    if r.returncode == 0 and r.stdout.strip():
        GPU_AVAILABLE = True
        print("GPU DETECTED:", r.stdout.strip())
except Exception as e:
    print("No nvidia-smi:", e)

if not GPU_AVAILABLE:
    print("FATAL: No NVIDIA GPU. Runtime -> Change runtime type -> Hardware accelerator -> GPU.")
    print("Refusing to run computationally expensive steps on CPU.")
    # raise SystemExit(1)

PROJECT = "/content/rso-protein-design-exploration"
if not Path(PROJECT).exists():
    get_ipython().system('git clone --depth 1 https://github.com/sokrypton/ColabDesign.git 2>&1 | tail -5')
    # Clone YOUR repo (replace URL with your fork if needed)
    # get_ipython().system('git clone https://github.com/YOUR_USER/rso-protein-design-exploration.git {PROJECT} 2>&1 | tail -3')
    print("Clone your project repo if needed; current minimal setup uses configs from src.")

sys.path.insert(0, "/content/ColabDesign")

# Install ColabDesign deps (Colab environment usually has many already)
get_ipython().run_line_magic('pip', 'install -q \
  git+https://github.com/sokrypton/ColabDesign.git@main biopython tqdm pyyaml pandas matplotlib seaborn 2>&1 | tail -5')

# --- AlphaFold params (official 2022-12-06 release, ~3.6 GB) + Zhang TMscore binary ---
# ColabDesign looks for params at ./params relative to CWD; Colab kernel CWD is /content.
os.chdir("/content")
if not os.path.isdir("params") or not any(f.endswith(".npz") for f in os.listdir("params")):
    get_ipython().system('mkdir -p params')
    get_ipython().system('apt-get install -qq aria2')
    get_ipython().system('aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar')
    get_ipython().system('tar -xf alphafold_params_2022-12-06.tar -C params')
print("AF2 params:", sorted(f for f in os.listdir("params") if f.endswith(".npz")))
if not os.path.isfile("TMscore"):
    get_ipython().system('wget -qnc https://zhanggroup.org/TM-score/TMscore.cpp')
    get_ipython().system('g++ -static -O3 -ffast-math -lm -o TMscore TMscore.cpp')
print("TMscore binary present:", os.path.isfile("TMscore"))


## Imports and reproducibility setup

We load the ColabDesign `mk_afdesign_model` / `mk_mpnn_model` factories, fix seeds, and read `reproduction_100aa.yaml`.


In [ ]:
import os, json, random, hashlib, time
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())

# JAX >=0.11 compatibility shim for ColabDesign main (as of 2026-09,
# ColabDesign calls jax.lib.xla_bridge.get_backend() in clear_mem(), but
# jax.lib.xla_bridge was removed in JAX 0.10+/moved to jax._src.xla_bridge).
if not hasattr(jax.lib, "xla_bridge"):
    from jax._src import xla_bridge as _xla_bridge
    jax.lib.xla_bridge = _xla_bridge
print("xla_bridge backend:", jax.lib.xla_bridge.get_backend().platform)

# jnp.clip: JAX >=0.4.31 deprecated and 0.11 removed a_min/a_max kwargs (renamed min/max).
# ColabDesign vendored AlphaFold (modules.py / modules_multimer.py, 3 call sites)
# still calls jnp.clip(x, a_min=..., a_max=...). Wrap once; call sites resolve jnp.clip
# at call time via module globals, so this also covers already-imported modules.
import jax.numpy as jnp
if not getattr(jnp.clip, "_rso_compat", False):
    _orig_jnp_clip = jnp.clip
    def _clip_compat(arr, a_min=None, a_max=None, min=None, max=None):
        if a_min is not None:
            min = a_min
        if a_max is not None:
            max = a_max
        return _orig_jnp_clip(arr, min, max)
    _clip_compat._rso_compat = True
    jnp.clip = _clip_compat
print("jnp.clip a_min/a_max compat active:", jnp.clip(jnp.arange(5), a_max=2).tolist())
assert any("gpu" in str(d).lower() or "cuda" in str(d).lower() for d in jax.devices()), ("JAX sees no GPU.", jax.devices())

# Project local imports if repo cloned
sys.path.insert(0, str(Path(PROJECT) / "src"))
try:
    from rso_exploration.config import load_config, validate_config
    from rso_exploration.paths import ensure_run_dir, backbone_pdb_path, candidate_fasta_path, candidate_csv_path, predicted_pdb_path, metadata_path, status_path, loss_history_path
    from rso_exploration.provenance import collect_provenance, save_provenance
    print("Local rso_exploration package loaded.")
except Exception as e:
    print("Local package not available, using inline helpers (" + str(e) + ")")

# Config
cfg_path = Path(PROJECT) / "configs" / "reproduction_100aa.yaml"
if cfg_path.exists():
    cfg = load_config(cfg_path)
    errs = validate_config(cfg)
    print(f"Loaded config {cfg.experiment_name}: errors={errs}")
else:
    print("No config file found; using hard-coded reproduction defaults (matches configs/reproduction_100aa.yaml).")
    cfg = None

SEED = 42 if cfg is None else cfg.seed
random.seed(SEED); np.random.seed(SEED)


## Stage 1 — RSO Backbone Generation (100 aa, 100 iters)

**Biology context:** Instead of searching in discrete 20-letter sequence space (which makes gradients noisy and argmax "forgetting" the optimum), RSO propagates loss gradients into a *relaxed* (logit/PSSM-style) sequence representation, feeding the relaxed representation straight back into AlphaFold for the next step. Losses used here match the official notebook:

- radius of gyration (`rg`): 0.1 — encourages compact globular fold
- helix penalty (`helix`): -0.2 — reduce helical content to diversify
- contacts (`con`): 1.0 — encourage many internal contacts
- pLDDT confidence: 0.5 — encourage AF2-high-confidence predictions
- PAE confidence: 0.5 — low inter-residue predicted aligned error

Optimization runs in two phases: 90 iter normal + 10 iter with `save_best=True` (soft argmax convergence).

Note: the AlphaFold structure prediction network used for *design* is NOT an independent validator; this is the design engine itself. Self-consistency validation happens later: ProteinMPNN assigns discrete sequences, then a fresh AF2 prediction (same model family) is compared to the designed backbone. A truly independent validator (e.g. ESMFold) is not yet implemented (see README Q3).


In [ ]:
from colabdesign import mk_afdesign_model, clear_mem
from colabdesign.af.alphafold.common import residue_constants

def add_rg_loss(self, weight=0.1):
    def loss_fn(inputs, outputs):
        positions = outputs["structure_module"]["final_atom_positions"]
        ca = positions[:, residue_constants.atom_order["CA"]]
        center = ca.mean(0)
        rg = jnp.sqrt(jnp.square(ca - center).sum(-1).mean() + 1e-8)
        rg_th = 2.38 * ca.shape[0] ** 0.365
        return {"rg": jax.nn.elu(rg - rg_th)}
    self._callbacks["model"]["loss"].append(loss_fn)
    self.opt["weights"]["rg"] = weight

clear_mem()
LENGTH = 100 if cfg is None else cfg.rso.length
af_model = mk_afdesign_model(protocol="hallucination", loss_callback=None)
af_model.prep_inputs(length=LENGTH)
add_rg_loss(af_model, 0.1 if cfg is None else cfg.rso.loss.rg_weight)

# Weights
if cfg is not None:
    w = cfg.rso.loss
    af_model.opt["weights"]["helix"] = w.helix_weight
    af_model.opt["weights"]["con"]   = w.con_weight
    af_model.opt["weights"]["plddt"] = w.plddt_weight
    af_model.opt["weights"]["pae"]   = w.pae_weight
else:
    af_model.opt["weights"]["helix"] = -0.2
    af_model.opt["weights"]["con"]   = 1.0
    af_model.opt["weights"]["plddt"] = 0.5
    af_model.opt["weights"]["pae"]   = 0.5

print("weights:", af_model.opt["weights"])
print(f"Starting RSO (L={LENGTH}) — compiling JAX first step may take 1-5 min ...")

af_model.restart(mode=["gumbel", "soft"], rm_aa="C")

stage1 = 90 if cfg is None else cfg.rso.stage1_iterations
stage2 = 10 if cfg is None else cfg.rso.stage2_iterations
t0 = time.time()
af_model.design_logits(stage1)
af_model.design_logits(stage2, save_best=True)
rso_time = time.time() - t0
print(f"RSO DONE in {rso_time:.1f}s")

# Save results
run_id = f"repro_100aa_s{SEED}_" + datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
rd = ensure_run_dir(run_id) if 'ensure_run_dir' in dir() else Path("/content/results/runs") / run_id; rd.mkdir(parents=True, exist_ok=True)
(rd/"stage1_rso").mkdir(exist_ok=True)

bb_id = "bb0"
pdb_out = backbone_pdb_path(rd, bb_id) if 'backbone_pdb_path' in dir() else rd/"stage1_rso"/f"{bb_id}.pdb"
af_model.save_pdb(str(pdb_out))
print(f"Saved backbone PDB -> {pdb_out}")

# Loss history — ColabDesign exposes the per-step design log via af_model.aux["log"]
# (not the af_model.log attribute). Read aux["log"] directly; fall back to the
# log attribute only for older ColabDesign versions that expose it.
_log_data = None
if "log" in (af_model.aux or {}):
    _log_data = af_model.aux["log"]
elif hasattr(af_model, "log"):
    _log_data = af_model.log
if _log_data is not None:
    ldf = pd.DataFrame(_log_data)
    ldf.index.name = "step"
    lp = loss_history_path(rd, bb_id) if 'loss_history_path' in dir() else rd/"stage1_rso"/f"{bb_id}_loss_history.csv"
    ldf.to_csv(lp)
    print(f"Loss history -> {lp} ({len(ldf)} rows)")
else:
    print("WARNING: no design log found in af_model.aux['log'] or af_model.log; loss_history.csv will be empty.")

# Best seq from RSO
best_seq = ""
try:
    seqs = af_model.get_seqs()
    best_seq = seqs[0] if isinstance(seqs, (list, tuple)) else str(seqs)
except Exception as e:
    print(f"(could not get RSO seq: {e})")

final_loss = float(np.mean(af_model.aux["losses"]["total"])) if 'total' in (af_model.aux.get("losses") or {}) else None

# Provenance and status
prov = collect_provenance(seed=SEED,
                          config_snapshot=cfg.to_dict() if cfg else None,
                          mpnn_weights="soluble")
prov.rso_runtime_seconds = rso_time
sp = metadata_path(rd) if 'metadata_path' in dir() else rd/"metadata.json"
save_provenance(prov, sp)
status = {"status": "stage2_pending",
          "rso_runtime_seconds": rso_time,
          "backbones": {bb_id: {"status": "rso_done", "final_loss": final_loss,
                                 "length": LENGTH, "seed": SEED,
                                 "best_rso_seq": best_seq}}}
(status_path(rd) if 'status_path' in dir() else rd/"status.json").write_text(json.dumps(status, indent=2))


## Stage 2 — ProteinMPNN Sequence Design

**Biology context:** The RSO process produces a *backbone geometry*, but the relaxed sequence representation is synthetic. We throw away the relaxed sequence and use ProteinMPNN (trained on PDB) to design 8 physically realistic amino-acid sequences that *ought* to fold into this backbone.

Settings per paper/official notebook:
- **weights:** soluble — designs biased toward soluble proteins (higher experimental success, slightly negative net charge)
- **temperature:** 0.1 — low diversity, low-energy sequences preferred
- **rm_aa:** `"C"` — cysteines excluded to avoid disulfide-scrambling artifacts


In [ ]:
from colabdesign.mpnn import mk_mpnn_model

NUM_SEQS = 8 if cfg is None else cfg.mpnn.num_seqs
TEMP    = 0.1 if cfg is None else cfg.mpnn.temperature
WEIGHTS = "soluble" if cfg is None else cfg.mpnn.weights
RM_AA   = "C" if cfg is None else cfg.mpnn.rm_aa

clear_mem()
mpnn_model = mk_mpnn_model(weights=WEIGHTS)
mpnn_model.prep_inputs(pdb_filename=str(pdb_out), chain="A", rm_aa=RM_AA)
t0 = time.time()
out = mpnn_model.sample(num=NUM_SEQS//8, batch=8, temperature=TEMP)
mpnn_time = time.time() - t0
print(f"ProteinMPNN sampled {len(out['seq'])} sequences in {mpnn_time:.1f}s")

cand_dir = rd/"stage2_mpnn"; cand_dir.mkdir(exist_ok=True)
# FASTA
fasta_p = candidate_fasta_path(rd, bb_id) if 'candidate_fasta_path' in dir() else cand_dir/f"{bb_id}_candidates.fasta"
with open(fasta_p, "w") as f:
    for i, (seq, score) in enumerate(zip(out["seq"], out["score"])):
        name = f"c{i}"
        f.write(f">{name} mpnn_score={float(score):.4f}\n{seq}\n")
print(f"FASTA -> {fasta_p}")

# CSV
csv_p = candidate_csv_path(rd, bb_id) if 'candidate_csv_path' in dir() else cand_dir/f"{bb_id}_candidates.csv"
pd.DataFrame({
    "candidate_id": [f"c{i}" for i in range(len(out["seq"]))],
    "seq": out["seq"],
    "score": [float(s) for s in out["score"]],
    "temperature": [TEMP]*len(out["seq"]),
    "weights": [WEIGHTS]*len(out["seq"]),
}).to_csv(csv_p, index=False)

# Update status
status_path_obj = status_path(rd) if 'status_path' in dir() else rd/"status.json"
st = json.loads(status_path_obj.read_text())
st["stage2_mpnn"] = {"num_seqs": NUM_SEQS, "temperature": TEMP,
                       "weights": WEIGHTS, "rm_aa": RM_AA,
                       "mpnn_runtime_seconds": mpnn_time}
st["status"] = "stage3_pending"
status_path_obj.write_text(json.dumps(st, indent=2))
print(f"CSV   -> {csv_p}")


## Stage 3 — Single-Sequence AF2 Self-Consistency Validation

**Biology context:** A design is only useful if a structure predictor (given only the designed amino-acid sequence, NOT the backbone) folds the sequence into something structurally close to the designed backbone. We use AlphaFold model_4_ptm in single-sequence mode (3 recycles, no templates, no MSA) to re-predict each of the 8 candidates. Because the RSO design engine is also AlphaFold-derived, this is a **self-consistency** check between design and evaluation models from the same family, not an independent ground truth.

We then compute structural agreement of each prediction vs. the designed backbone:
- **RMSD (Å)** — per-Cα root mean square deviation after optimal superposition (lower = better)
- **TM-score** — template modeling score 0..1 (length-invariant; >0.5 typically implies same fold)
- **mean pLDDT** — per-residue AF2 confidence 0..100 (higher = more confident locally)
- **pTM** — AF2 predicted TM score 0..1 (if available from model_ptm output)

Important caveats:
- High pLDDT ≠ experimental success (expression/solubility/stability).
- The same AF2 family participated in *both* design (RSO loss) and *evaluation* — this can produce method-related biases.


In [ ]:
# Stage 3 - independent AF2 single-sequence validation (per official designability_test)
import subprocess
from colabdesign.af.alphafold.common import residue_constants
clear_mem()

AF_MODEL_NAME = "model_4_ptm" if cfg is None else cfg.validation.alphafold_model
NUM_RECYCLES = 3 if cfg is None else cfg.validation.num_recycles
TMSCORE_BIN = Path("/content/TMscore")

af_val = mk_afdesign_model(protocol="fixbb", best_metric="rmsd", use_templates=False)
af_val.prep_inputs(pdb_filename=str(pdb_out), chain="A")
af_val.restart(rm_aa="C")

def read_ca_xyz(pdb_path):
    xyz = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith("ATOM  ") and line[12:16].strip() == "CA":
                xyz.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.asarray(xyz, dtype=float)

def kabsch_align(mod_ca, ref_ca):
    """Optimal rigid-body superposition of mod onto ref (Nx3 CA coords)."""
    P = mod_ca - mod_ca.mean(axis=0)
    q_mean = ref_ca.mean(axis=0)
    Q = ref_ca - q_mean
    H = P.T @ Q
    U, _, Vt = np.linalg.svd(H)
    D = np.diag([1.0, 1.0, np.sign(np.linalg.det(Vt.T @ U.T))])
    R = Vt.T @ D @ U.T
    return P @ R.T + q_mean

def aligned_ca_rmsd(mod_ca, ref_ca):
    n = min(len(mod_ca), len(ref_ca))
    a = kabsch_align(mod_ca[:n], ref_ca[:n])
    d = ref_ca[:n] - a
    return float(np.sqrt(np.mean(np.sum(d * d, axis=-1))))

def tm_score_aligned(mod_ca, ref_ca):
    """Zhang TM-score after Kabsch superposition; d0 normalised by reference length."""
    n = min(len(mod_ca), len(ref_ca))
    a = kabsch_align(mod_ca[:n], ref_ca[:n])
    d0 = max(1.24 * (n - 15) ** (1/3) - 1.8, 0.5)
    d = np.sqrt(np.sum((ref_ca[:n] - a) ** 2, axis=-1))
    return float(np.sum(1.0 / (1.0 + (d / d0) ** 2)) / n)

def run_tmscore_binary(ref_pdb, pred_pdb):
    """Zhang TMscore executable -> (native-normalised TM-score, aligned RMSD)."""
    if not TMSCORE_BIN.is_file():
        return None, None
    try:
        proc = subprocess.run([str(TMSCORE_BIN), str(pred_pdb), str(ref_pdb)],
                              capture_output=True, text=True, timeout=120)
        tm_vals, rmsd_vals = [], []
        for line in proc.stdout.splitlines():
            s = line.strip()
            if s.startswith("TM-score") and "=" in s:
                try: tm_vals.append(float(s.split("=")[1].split()[0]))
                except Exception: pass
            elif s.startswith("RMSD") and "=" in s:
                try: rmsd_vals.append(float(s.split("=")[1].split()[0]))
                except Exception: pass
        tm_v = tm_vals[-1] if tm_vals else None    # Chain_2 (native/reference) normalisation
        rmsd_v = rmsd_vals[0] if rmsd_vals else None
        return tm_v, rmsd_v
    except Exception:
        return None, None

s3 = rd/"stage3_validation"; s3.mkdir(exist_ok=True)
rows = []
val_times = []
for i in range(len(out["seq"])):
    cid = f"c{i}"
    seq = out["seq"][i]
    t0 = time.time()
    try:
        af_val.predict(seq=seq, num_recycles=NUM_RECYCLES, num_models=1,
                       models=AF_MODEL_NAME, verbose=False)
        aux = af_val.aux
        log = aux.get("log", {})
        plddt_arr = np.array(aux.get("plddt", log.get("plddt"))).astype(float)
        if plddt_arr.size:
            mean_plddt = float(np.mean(plddt_arr))
            # ColabDesign get_plddt() averages bin centres in [0,1]; canonical AF pLDDT is 0-100
            if 0.0 <= mean_plddt <= 1.0:
                mean_plddt *= 100.0
                plddt_scale = "colabdesign_native_0_1_x100"
            else:
                plddt_scale = "native_0_100"
        else:
            mean_plddt, plddt_scale = None, None
        ptm = float(log["ptm"]) if "ptm" in log else None

        # save predicted PDB first (needed by binary + numpy metric paths)
        pred_p = predicted_pdb_path(rd, bb_id, cid)
        af_val.save_pdb(str(pred_p))

        # RMSD primary: ColabDesign log["rmsd"] (fixbb aligned CA RMSD = paper metric)
        rmsd = float(log["rmsd"]) if "rmsd" in log else None
        rmsd_source = "colabdesign_log" if rmsd is not None else None
        # TM-score + cross-check RMSD from official Zhang TMscore binary (Kabsch numpy fallback)
        tm, rmsd_bin = run_tmscore_binary(pdb_out, pred_p)
        if rmsd is None and rmsd_bin is not None:
            rmsd, rmsd_source = rmsd_bin, "tmscore_binary"
        ref_ca = read_ca_xyz(pdb_out)
        pred_ca = read_ca_xyz(pred_p)
        n = min(len(ref_ca), len(pred_ca))
        if rmsd is None and n > 15:
            rmsd = aligned_ca_rmsd(pred_ca, ref_ca)
            rmsd_source = "numpy_kabsch"
        if tm is None and n > 15:
            tm = tm_score_aligned(pred_ca, ref_ca)
            tm_source = "numpy_kabsch"
        elif tm is not None:
            tm_source = "tmscore_binary"
        else:
            tm_source = None

        vt = time.time() - t0
        val_times.append(vt)
        m_path = s3 / f"{bb_id}_{cid}_metrics.json"
        m = {"rmsd_angstrom": rmsd, "rmsd_source": rmsd_source,
             "tm_score": tm, "tm_source": tm_source,
             "mean_plddt": mean_plddt, "plddt_scale": plddt_scale, "ptm": ptm,
             "validation_runtime_seconds": vt,
             "validation_model": AF_MODEL_NAME,
             "af2_log": {k: (float(v) if isinstance(v, (int, float, np.floating)) else None)
                         for k, v in log.items()
                         if k in ("plddt", "ptm", "pae", "rmsd", "dgram_cce")}}
        m_path.write_text(json.dumps(m, indent=2))
        rows.append({"c": cid, "rmsd": rmsd, "tm": tm, "plddt": mean_plddt,
                     "ptm": ptm, "time_s": vt,
                     "rmsd_source": rmsd_source, "tm_source": tm_source,
                     "plddt_scale": plddt_scale})
        _r = "None" if rmsd is None else f"{rmsd:.3f}"
        _t = "None" if tm is None else f"{tm:.4f}"
        _p = "None" if mean_plddt is None else f"{mean_plddt:.2f}"
        print(f"  {cid}: RMSD={_r} ({rmsd_source})  TM={_t} ({tm_source})  mean-pLDDT={_p}  ({vt:.1f}s)")
    except Exception as e:
        vt = time.time() - t0
        err_path = s3 / f"{bb_id}_{cid}_metrics.json"
        err_path.write_text(json.dumps({"error": str(e)[:500], "validation_runtime_seconds": vt}, indent=2))
        rows.append({"c": cid, "error": str(e)[:200]})
        print(f"  {cid}: FAILED: {str(e)[:200]}")

st2 = json.loads(status_path_obj.read_text())
st2["status"] = "completed"
st2["validation"] = {"model": AF_MODEL_NAME, "num_recycles": NUM_RECYCLES,
                     "total_validation_seconds": sum(val_times)}
st2["backbones"][bb_id]["candidates"] = rows
status_path_obj.write_text(json.dumps(st2, indent=2, default=str))
print("Done.")


## Stage 4 — Ranking and Save Metrics

We rank candidates independently by each metric (never sum arbitrary units). Report:
- rank by lowest RMSD
- rank by highest TM-score
- rank by highest mean pLDDT

Then save unified metrics to CSV for downstream collection (`scripts/collect_metrics.py`).


In [ ]:
# Summarize ranking
results = pd.DataFrame([r for r in rows if "rmsd" in r])
if not results.empty:
    results["rank_rmsd"] = results["rmsd"].rank(ascending=True, method="min")
    results["rank_tm"]    = results["tm"].rank(ascending=False, method="min")
    results["rank_plddt"] = results["plddt"].rank(ascending=False, method="min")
    display(results.sort_values("rank_rmsd").style.set_caption("Independent ranking (lower rank number = better)."))
    # Collect per-candidate CSV via project script (if cloned)
    collect_script = Path(PROJECT)/"scripts"/"collect_metrics.py"
    if collect_script.exists():
        print(f"Running {collect_script} ...")
        r = subprocess.run([sys.executable, str(collect_script)], capture_output=True, text=True)
        print(r.stdout)
        if r.stderr.strip(): print("STDERR:", r.stderr)
else:
    print("No successful candidates to rank.")


## Download results

Save the results archive (runs + metrics + configs + provenance) and download locally to merge into the repository, then run `scripts/verify_outputs.py` and `scripts/make_figures.py`.


In [ ]:
# Zip + download (Colab UI will prompt)
import shutil, datetime
archive = "/content/rso_results_" + datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ") + ".zip"
base = str(Path(rd).parent.parent)  # results/ level
print("Archiving", base, "->", archive)
shutil.make_archive(archive.replace(".zip",""), 'zip', base)
print(f"Wrote {archive} ({Path(archive).stat().st_size/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print("Not in Colab UI; download archive via file browser or mount Google Drive.")
